# Unified CONDOR/FlexDC inference + evaluation notebook

This notebook is for running the new unified prediction, optimization, and end-to-end evaluation scripts in Google Colab.

It supports all four trained model variants:

- `condor / normal`
- `condor / raw`
- `flexdc / normal`
- `flexdc / raw`

**Run/skip rule:** cells marked **Colab only** should be run in Google Colab. Cells marked **Local optional** are for reference if you copy commands back to your PC.


## 1. Environment controls — RUN in Colab

Edit the repo URLs before running. If your repositories are private, use a GitHub token or mount Drive and copy files manually.


In [ ]:
from pathlib import Path
import os

RUN_ENV = "colab"  # expected: "colab" for this notebook
WORKSPACE = Path("/content/workspace")

# TODO: replace these with your actual repo URLs and branches.
COMDER_REPO_URL = "PUT_COMDER_REPO_URL_HERE"
FLEXDC_REPO_URL = "PUT_FLEXDC_REPO_URL_HERE"
COMDER_BRANCH = "main"
FLEXDC_BRANCH = "main"

# Device: use cuda in Colab GPU runtime; cpu also works.
DEVICE = "cuda"

# W&B controls. Set WANDB_MODE="disabled" to turn logging off.
WANDB_ENTITY = "amenon06-boston-university"
WANDB_PROJECT = "flexdc-unified-inference"
WANDB_MODE = "online"

print("RUN_ENV:", RUN_ENV)
print("WORKSPACE:", WORKSPACE)


## 2. Install minimal dependencies — RUN in Colab

Do not install the whole repo requirements file unless needed. Colab already includes PyTorch in most runtimes.


In [ ]:
%pip install -q wandb pandas numpy scipy scikit-learn tqdm matplotlib tabulate

import torch
print("Torch:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())
if DEVICE == "cuda" and not torch.cuda.is_available():
    print("WARNING: DEVICE is cuda but torch.cuda.is_available() is False. Change DEVICE to 'cpu'.")


## 3. Clone repositories — RUN in Colab

Skip this cell only if you already copied both repos into `/content/workspace`.


In [ ]:
!rm -rf "$WORKSPACE"
!mkdir -p "$WORKSPACE"

!git clone --branch "$COMDER_BRANCH" "$COMDER_REPO_URL" "$WORKSPACE/comder-main"
!git clone --branch "$FLEXDC_BRANCH" "$FLEXDC_REPO_URL" "$WORKSPACE/flexdc-sim"


## 4. Define paths — RUN after clone/copy

This cell makes imports work and defines where the model/data/scripts live.


In [ ]:
import sys

COMDER_ROOT = WORKSPACE / "comder-main"
FLEXDC_ROOT = WORKSPACE / "flexdc-sim"
AM_FLEXDC_ROOT = COMDER_ROOT / "am_flexdc"
TRAIN_DIR = AM_FLEXDC_ROOT / "train"
PILOT_DIR = AM_FLEXDC_ROOT / "data" / "pilots" / "traditionaliso_newqos_pilot"
MODELS_DIR = AM_FLEXDC_ROOT / "models"
RESULTS_DIR = AM_FLEXDC_ROOT / "results" / "unified_eval_runs"
RESULTS_DIR.mkdir(parents=True, exist_ok=True)

if str(TRAIN_DIR) not in sys.path:
    sys.path.insert(0, str(TRAIN_DIR))

RESULTS_CSV = PILOT_DIR / "traditional_iso16_newqos_AQA_combined_grid_search_results.csv"
DIAGNOSTICS_CSV = PILOT_DIR / "traditional_iso16_newqos_AQA_combined_grid_search_diagnostics.csv"

print("TRAIN_DIR:", TRAIN_DIR)
print("PILOT_DIR:", PILOT_DIR)
print("RESULTS_DIR:", RESULTS_DIR)


## 5. Optional Google Drive dataset/model copy — RUN only if data/model are not in GitHub

If the combined pilot CSVs or model checkpoints are too large for GitHub, mount Drive and copy them into the expected folders.

Skip this cell if the files are already in the repo.


In [ ]:
# from google.colab import drive
# drive.mount('/content/drive')
#
# PILOT_DIR.mkdir(parents=True, exist_ok=True)
# !cp "/content/drive/MyDrive/path/to/traditional_iso16_newqos_AQA_combined_grid_search_results.csv" "$PILOT_DIR/"
# !cp "/content/drive/MyDrive/path/to/traditional_iso16_newqos_AQA_combined_grid_search_diagnostics.csv" "$PILOT_DIR/"
#
# MODELS_DIR.mkdir(parents=True, exist_ok=True)
# !cp -r "/content/drive/MyDrive/path/to/models/*" "$MODELS_DIR/"


## 6. Required-file check — RUN before any inference

This catches missing scripts, datasets, or checkpoints before the long FlexDC run starts.


In [ ]:
required = [
    TRAIN_DIR / "data_center_model.py",
    TRAIN_DIR / "am_unified_training_utilities.py",
    TRAIN_DIR / "am_unified_predict_one.py",
    TRAIN_DIR / "am_unified_optimize_one.py",
    TRAIN_DIR / "am_unified_end_to_end_eval.py",
    RESULTS_CSV,
    DIAGNOSTICS_CSV,
    FLEXDC_ROOT / "src" / "peacsim" / "am_data_extraction_wizard.py",
]
missing = [str(p) for p in required if not p.exists()]
if missing:
    print("Missing required files:")
    for p in missing:
        print(" -", p)
    raise FileNotFoundError("Fix missing files before continuing.")
print("All required files found.")


## 7. W&B login — RUN if WANDB_MODE is online/offline

Skip if `WANDB_MODE = "disabled"`.


In [ ]:
if WANDB_MODE != "disabled":
    import wandb
    wandb.login()
else:
    print("W&B disabled.")


## 8. Choose model variant and checkpoint — RUN

Change these controls to test a different trained model.

For example:
- `TARGET_FAMILY="condor"`, `TARGET_MODE="normal"`
- `TARGET_FAMILY="condor"`, `TARGET_MODE="raw"`
- `TARGET_FAMILY="flexdc"`, `TARGET_MODE="normal"`
- `TARGET_FAMILY="flexdc"`, `TARGET_MODE="raw"`


In [ ]:
TARGET_FAMILY = "condor"   # "condor" or "flexdc"
TARGET_MODE = "normal"     # "normal" or "raw"
RAW_QOS_AGGREGATION = "mean"
USE_NORM_COST = "auto"     # auto means CONDOR=True, FlexDC=False
USE_NORM_PR = "true"       # should match training default

MODEL_FILE = MODELS_DIR / f"{TARGET_FAMILY}_{TARGET_MODE}" / f"am_{TARGET_FAMILY}_{TARGET_MODE}_newqos_v1_state_dict.pt"

# Override manually if your checkpoint has a different name:
# MODEL_FILE = MODELS_DIR / "condor_normal" / "your_checkpoint.pt"

if not MODEL_FILE.exists():
    raise FileNotFoundError(f"Model checkpoint not found: {MODEL_FILE}")

print("Model variant:", TARGET_FAMILY, TARGET_MODE)
print("Model file:", MODEL_FILE)


## 9. Choose test scenario — RUN

This is the model/FlexDC context to evaluate. Starting values do not have to be optimal; they just seed gradient descent.


In [ ]:
WORKLOAD_CONFIG = FLEXDC_ROOT / "configs" / "workload" / "W2-short-qos5555.ini"
EXPERIMENT_CONFIG = FLEXDC_ROOT / "configs" / "experiment" / "new_iso" / "traditional_signal" / "generated_server_counts" / "exp_traditional_iso16_servers_1000.ini"
SERVER_COUNT = 1000
UTILIZATION = 0.80

START_PBAR = 0.4858666666666666
START_R = 0.2225185185185185
START_WEIGHTS = "0.25,0.25,0.25,0.25"

ITERATIONS = 150
LR = 0.01

# For CONDOR normal, released example weights are 0.05,0.7,2.0.
# For FlexDC normal, auto defaults to 1,1,1 because the targets already include beta/psi weights.
OBJECTIVE_WEIGHTS = "auto"

print("Workload config:", WORKLOAD_CONFIG)
print("Experiment config:", EXPERIMENT_CONFIG)


## 10. Compile scripts — RUN

This checks syntax before running.


In [ ]:
%cd {TRAIN_DIR}
!python -m py_compile \
  am_unified_predict_one.py \
  am_unified_optimize_one.py \
  am_unified_end_to_end_eval.py


## 11. Predict one point — OPTIONAL model-only test

Run this to check that the checkpoint can produce a prediction for the starting configuration. This does **not** run FlexDC.


In [ ]:
%cd {TRAIN_DIR}

PREDICT_OUT = RESULTS_DIR / f"predict_one_{TARGET_FAMILY}_{TARGET_MODE}.json"

!python am_unified_predict_one.py \
  --model-file "{MODEL_FILE}" \
  --norm-source-results-csv "{RESULTS_CSV}" \
  --workload-config "{WORKLOAD_CONFIG}" \
  --experiment-config "{EXPERIMENT_CONFIG}" \
  --target-family "{TARGET_FAMILY}" \
  --target-mode "{TARGET_MODE}" \
  --raw-qos-aggregation "{RAW_QOS_AGGREGATION}" \
  --use-norm-cost "{USE_NORM_COST}" \
  --use-norm-pr "{USE_NORM_PR}" \
  --server-count "{SERVER_COUNT}" \
  --utilization "{UTILIZATION}" \
  --pbar-kw-per-server "{START_PBAR}" \
  --r-kw-per-server "{START_R}" \
  --weights "{START_WEIGHTS}" \
  --objective-weights "{OBJECTIVE_WEIGHTS}" \
  --device "{DEVICE}" \
  --out-json "{PREDICT_OUT}" \
  --wandb-project "{WANDB_PROJECT}" \
  --wandb-entity "{WANDB_ENTITY}" \
  --wandb-run-name "predict-one-{TARGET_FAMILY}-{TARGET_MODE}" \
  --wandb-mode "{WANDB_MODE}"


## 12. Optimize only — OPTIONAL dry run

Run this before full FlexDC validation if you only want the surrogate-selected candidate. This does **not** run FlexDC.


In [ ]:
%cd {TRAIN_DIR}

OPT_DIR = RESULTS_DIR / f"optimize_only_{TARGET_FAMILY}_{TARGET_MODE}"

!python am_unified_optimize_one.py \
  --model-file "{MODEL_FILE}" \
  --norm-source-results-csv "{RESULTS_CSV}" \
  --workload-config "{WORKLOAD_CONFIG}" \
  --experiment-config "{EXPERIMENT_CONFIG}" \
  --target-family "{TARGET_FAMILY}" \
  --target-mode "{TARGET_MODE}" \
  --raw-qos-aggregation "{RAW_QOS_AGGREGATION}" \
  --use-norm-cost "{USE_NORM_COST}" \
  --use-norm-pr "{USE_NORM_PR}" \
  --server-count "{SERVER_COUNT}" \
  --utilization "{UTILIZATION}" \
  --start-pbar-kw-per-server "{START_PBAR}" \
  --start-r-kw-per-server "{START_R}" \
  --start-weights "{START_WEIGHTS}" \
  --iterations "{ITERATIONS}" \
  --lr "{LR}" \
  --objective-weights "{OBJECTIVE_WEIGHTS}" \
  --device "{DEVICE}" \
  --out-dir "{OPT_DIR}" \
  --wandb-project "{WANDB_PROJECT}" \
  --wandb-entity "{WANDB_ENTITY}" \
  --wandb-run-name "optimize-one-{TARGET_FAMILY}-{TARGET_MODE}" \
  --wandb-mode "{WANDB_MODE}"


## 13. Full end-to-end evaluation — RUN when ready

This runs optimization, then runs FlexDC twice: starting configuration and selected configuration.

Skip this if the FlexDC repo/dependencies are not installed in the Colab runtime.


In [ ]:
%cd {TRAIN_DIR}

E2E_DIR = RESULTS_DIR / f"e2e_{TARGET_FAMILY}_{TARGET_MODE}_W2_N1000_U080"

!python am_unified_end_to_end_eval.py \
  --model-file "{MODEL_FILE}" \
  --norm-source-results-csv "{RESULTS_CSV}" \
  --workload-config "{WORKLOAD_CONFIG}" \
  --experiment-config "{EXPERIMENT_CONFIG}" \
  --target-family "{TARGET_FAMILY}" \
  --target-mode "{TARGET_MODE}" \
  --raw-qos-aggregation "{RAW_QOS_AGGREGATION}" \
  --use-norm-cost "{USE_NORM_COST}" \
  --use-norm-pr "{USE_NORM_PR}" \
  --server-count "{SERVER_COUNT}" \
  --utilization "{UTILIZATION}" \
  --start-pbar-kw-per-server "{START_PBAR}" \
  --start-r-kw-per-server "{START_R}" \
  --start-weights "{START_WEIGHTS}" \
  --iterations "{ITERATIONS}" \
  --lr "{LR}" \
  --objective-weights "{OBJECTIVE_WEIGHTS}" \
  --device "{DEVICE}" \
  --flexdc-root "{FLEXDC_ROOT}" \
  --flexdc-python python \
  --run-flexdc \
  --out-dir "{E2E_DIR}" \
  --wandb-project "{WANDB_PROJECT}" \
  --wandb-entity "{WANDB_ENTITY}" \
  --wandb-run-name "e2e-{TARGET_FAMILY}-{TARGET_MODE}-W2-N1000-U080" \
  --wandb-mode "{WANDB_MODE}"


## 14. Inspect and download outputs — RUN after evaluation


In [ ]:
import pandas as pd
from google.colab import files

if 'E2E_DIR' in globals() and E2E_DIR.exists():
    summary_path = E2E_DIR / "end_to_end_validation_summary.csv"
    report_path = E2E_DIR / "end_to_end_validation_report.md"
    if summary_path.exists():
        display(pd.read_csv(summary_path))
    if report_path.exists():
        print(report_path.read_text())

    zip_path = Path('/content') / f"{E2E_DIR.name}.zip"
    !zip -r "{zip_path}" "{E2E_DIR}"
    files.download(str(zip_path))
else:
    print("No E2E_DIR found yet. Run the full end-to-end cell first.")
